# Adding ByteTrack MOT Functionality

In [1]:
%pip -V

pip 25.3 from /Users/rjunw/Desktop/dev/nba2nba/.venv/lib/python3.11/site-packages/pip (python 3.11)
Note: you may need to restart the kernel to use updated packages.


## Installs + Imports

In [22]:
import os
import youtube_dl

import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import ListedColormap
from IPython.display import Video, clear_output, HTML

import torch
import cv2
import supervision as sv
from PIL import Image


from bytetrack.byte_tracker import BYTETracker
from ultralytics import YOLO
from rfdetr import RFDETRMedium
from court import Court

from tqdm import tqdm

# model vars
YOLOv11x_CHECKPOINT = "../models/weights/yolov11x_kpd/checkpoint_best_total.pt" # 300 epochs, default config
RF_DETR_MEDIUM_CHECKPOINT = "../models/weights/rf_detr_od/checkpoint_best_total.pth" # 50 epochs, bs 16, grad_accum 1

# dataset paths
player_od_data = "../data/.cache/bball_od"
court_kpd_data = "../data/.cache/bball_court_kpd"

## Helpers

In [3]:
def get_vid_url(link, format_note='1080p'):
    # get video url from youtube
    yt_url = link
    ydl = youtube_dl.YoutubeDL()
    info_dict = ydl.extract_info(yt_url, download=False)
    formats = info_dict.get('formats',None)
    video_url = None
    for f in formats:
        if f['format_note'] == format_note:
            video_url = f['url']
            break
        
    return video_url

In [4]:
def get_frames(vid_url, frame_skip=4, max_seconds=10):
    """
    Input video url
    Output frames
    """

    print(f"###### GETTING FROMS FROM {vid_url} ######")

    # get video as input frames
    cap = cv2.VideoCapture(url)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps
    print(f"FRAMES: {frame_count}")
    print(f"DURATION (s): {duration}")

    # every skip_frames append to frames
    frames = []
    if cap.isOpened():
        timestep = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                print("VIDEO FINISHED")
                break 
            if len(frames) * frame_skip > max_seconds * fps:
                print("MAX TIME REACHED")
                break
            
            if timestep % frame_skip == 0:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
            timestep += 1
        cap.release()
    cv2.destroyAllWindows()
    frames = np.asarray(frames)   
    return frames, frame_count, fps

## Setup

In [5]:
od_model = RFDETRMedium(pretrain_weights=RF_DETR_MEDIUM_CHECKPOINT)
od_model.optimize_for_inference() # compile model for faster inference
kpd_model = YOLO(YOLOv11x_CHECKPOINT, task='pose')

Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


num_classes mismatch: pretrain weights has 5 classes, but your model has 90 classes
reinitializing detection head with 5 classes


Loading pretrain weights


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


In [6]:
# defining a dataset iterator
ds = sv.DetectionDataset.from_coco(
    images_directory_path=f"{player_od_data}/test",
    annotations_path=f"{player_od_data}/test/_annotations.coco.json",
)

## ByteTrack with RF-DETR Detector

First, let's download a youtube video we want to track.

In [7]:
# 1080p can get too large
url = get_vid_url("https://www.youtube.com/watch?v=LgTqkEkNJrE", "720p")

# display video
Video(url)

[youtube] LgTqkEkNJrE: Downloading webpage


[youtube] LgTqkEkNJrE: Downloading ANDROID API JSON


In [8]:
vid_config = {
    'frame_skip': 2,
    'max_seconds': 10
}
vid_frames, vid_frame_count, vid_fps = get_frames(url, **vid_config)
vid_frames.shape

###### GETTING FROMS FROM https://rr3---sn-8xgp1vo-xfge.googlevideo.com/videoplayback?expire=1764840878&ei=TgExacbkI73HkucP4KTu6Ak&ip=2600%3A4041%3A537c%3A6b00%3A51c0%3Ac4c8%3A3901%3Adf55&id=o-AAQU05_DAQq31kcDGqi7BCfiwrLopukK64yWyQcillAE&itag=398&source=youtube&requiressl=yes&xpc=EgVo2aDSNQ%3D%3D&met=1764819278%2C&mh=3p&mm=31%2C26&mn=sn-8xgp1vo-xfge%2Csn-p5qs7n6d&ms=au%2Conr&mv=m&mvi=3&pl=38&rms=au%2Cau&initcwndbps=3473750&bui=AdEuB5Sx-6MdvzVdFOis67lM9t76BrbEiYB8_R3XmRTSLgy3cv5h9Sb57WPHac3SeVFQwzU0VrdmxJgu&spc=6b0G_OvGCTj8&vprv=1&svpuc=1&mime=video%2Fmp4&rqh=1&gir=yes&clen=4820263&dur=37.833&lmt=1726705768169085&mt=1764818932&fvip=1&keepalive=yes&fexp=51552689%2C51565116%2C51565682%2C51580968&c=ANDROID&txp=4537434&sparams=expire%2Cei%2Cip%2Cid%2Citag%2Csource%2Crequiressl%2Cxpc%2Cbui%2Cspc%2Cvprv%2Csvpuc%2Cmime%2Crqh%2Cgir%2Cclen%2Cdur%2Clmt&sig=AJfQdSswRQIhANSk3ZrsgUpWJzEdkKAwWKjpPHHqutjuITd7b58VcE3yAiA5isJ5JILBocYvAWrklituY1RDGToL-a8OnstGlpYbfA%3D%3D&lsparams=met%2Cmh%2Cmm%2Cmn%2Cms%

(151, 720, 1280, 3)

Recall normal detection

### OD Video

In [9]:
# vid_frames = torch.Tensor(vid_frames)
i = 1
od_frames = []
for image in tqdm(vid_frames):
    # preprocessing
    # image = image/255
    # image = cv2.resize(image, (640, 640))
    # image = torch.tensor(image, dtype=torch.float32)
    # image = image.permute(2, 0, 1)
    image_array = cv2.resize(image, (640, 640))
    image = Image.fromarray(image_array)


    # inference
    od_detections = od_model.predict(image)
    kpd_detections = kpd_model(image)

    # plotting schtuff
    image_court = Court()
    text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
    thickness = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
    color = sv.ColorPalette.from_hex([
        "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
        "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
    ])

    # OD 
    bbox_annotator = sv.BoxAnnotator(color=color,thickness=thickness)
    label_annotator = sv.LabelAnnotator(
        color=color,
        text_color=sv.Color.BLACK,
        text_scale=text_scale)

    detections_labels = [
        f"{ds.classes[class_id]} {confidence:.2f}"
        for class_id, confidence
        in zip(od_detections.class_id, od_detections.confidence)
    ]

    detections_image = image.copy()
    detections_image = bbox_annotator.annotate(detections_image, od_detections)
    detections_image = label_annotator.annotate(detections_image, od_detections, detections_labels)

    # KPD
    # get kp object via supervision
    key_points = sv.KeyPoints.from_ultralytics(kpd_detections[0])
    try:
        # filter out low confidence key points
        kp_to_keep = key_points.confidence > 0.1

        # build a filtered key point object
        kp_ids = np.array([i for i in range(key_points.confidence.shape[1])])[kp_to_keep.flatten()]

        # filter down edge list to what we have
        image_court.kp_edge_list = [(edge[0]+1, edge[1]+1) for edge in image_court.kp_edge_list if edge[0] in set(kp_ids) and edge[1] in set(kp_ids)]


        kp_thresholded = key_points.xy[kp_to_keep]
        kp_confidence_thresholded = key_points.confidence[kp_to_keep]
        thresholded_key_points = sv.KeyPoints(
            xy=kp_thresholded.reshape(-1,*kp_thresholded.shape),
            class_id=key_points.class_id,
            confidence=kp_confidence_thresholded.reshape(-1,*kp_confidence_thresholded.shape)
        )
    except:
        thresholded_key_points = key_points

    # just plotting stuff
    kpd_image = image_array

    # annotate keypoints
    vertex_annotator = sv.VertexAnnotator(radius=3, color=sv.Color.YELLOW)
    kpd_image = vertex_annotator.annotate(
        scene=kpd_image,
        key_points=thresholded_key_points)

    # annotate edges
    edge_annotator = sv.EdgeAnnotator(thickness=2, edges=image_court.kp_edge_list)
    kpd_image = edge_annotator.annotate(
        scene=kpd_image,
        key_points=key_points)

    kpd_image = sv.resize_image(
        kpd_image,
        resolution_wh=(640, 640), # keep same as model input size for easier viz
        keep_aspect_ratio=True
    )

    # convert to RGB
    kpd_image = cv2.cvtColor(kpd_image, cv2.COLOR_BGR2RGB)

    if i % 20 == 0:
        sv.plot_images_grid(images=[detections_image, kpd_image], grid_size=(1, 2), titles=["Possession", "Court"])
   
    od_frames.append(np.array(detections_image))
    clear_output()
    i += 1

100%|██████████| 151/151 [01:01<00:00,  2.46it/s]


In [10]:
fig = plt.figure()
im = plt.imshow(od_frames[0])
plt.close()

def init():
    im.set_data(od_frames[0])

def animate(i):
    im.set_data(od_frames[i])
    return im

anim = animation.FuncAnimation(fig, animate, init_func=init, frames=len(od_frames),
                               interval=vid_fps*vid_config['frame_skip'])
HTML(anim.to_html5_video())

Immediately, with a full video, we can see some limitations of our model, especially the keypoint model. There are typically many perspectives in a game, and the model struggles to identify proper keypoints in shifted views. 

- May require more augmentations (warping, rotation, etc.)
- Also potentially just a limitation of the training data

### Tracking

We can output the bounding boxes to bytetrack and use it to track the objects.

In [11]:
class BTArgs:
    def __init__(self):
        self.track_thresh = 0.5
        self.track_buffer = 30
        self.match_thresh = 0.8
        self.aspect_ratio_thresh = 1.6
        self.min_box_area = 10
        self.mot20 = False

tracker = BYTETracker(BTArgs(), frame_rate=30)


In [ ]:
width_px = 640
height_px = 640
dpi = 100
figsize_width_inches = width_px / dpi
figsize_height_inches = height_px / dpi

color_map = {}
cmap = plt.cm.gist_rainbow
tracked_frames = []
frame_idx = 0
for image in tqdm(vid_frames):
    orig_image = np.copy(image)
    # preprocessing
    image = image/255
    image = cv2.resize(image, (640, 640))
    image = torch.tensor(image, dtype=torch.float32)
    image = image.permute(2, 0, 1)
    
    # image_array = cv2.resize(image, (640, 640))
    # image = Image.fromarray(image_array)


    # inference
    od = od_model.predict(image)

    image = image.unsqueeze(0)
    kpd = kpd_model(image)

    od_track = np.concatenate([od.xyxy, od.confidence.reshape(-1, 1)], axis=1)

    # tracking
    online_tracker = tracker.update(od_track,  image.shape[2:], image.shape[2:])

    # plotting - might wanna do a homography on circles to make it nicer
    fig = plt.figure(figsize=(figsize_width_inches, figsize_height_inches))
    fig.add_axes([0, 0, 1, 1])
    plt.imshow(od_frames[frame_idx]) # overlay tracker on bbox viz from before
    for track in online_tracker:
        track_id = track.track_id
        x1, y1, x2, y2 = track.tlbr
        score = track.score

        if track_id not in color_map:
            color_map[track_id] = list(cmap(np.random.rand()))
            color_map[track_id][3] = 1

        color = color_map[track_id]
        mid_x = (x1 + x2) / 2
        plt.scatter(mid_x, y2, color=color)
        plt.text(mid_x, y2+10, f"ID: {track_id}\nScore: {score:.2f}", color=color, ha='center', va='top', fontsize=14)
    plt.axis('off')
    fig.canvas.draw()
    data = np.array(fig.canvas.renderer.buffer_rgba())
    tracked_frames.append(data)
    plt.close()
    clear_output()

    # if frame_idx == 5:
    #     break

    frame_idx += 1
    


100%|██████████| 151/151 [01:06<00:00,  2.27it/s]


In [145]:
fig = plt.figure(figsize=(figsize_width_inches, figsize_height_inches))
fig.add_axes([0, 0, 1, 1])
im = plt.imshow(tracked_frames[0])
plt.close()

def init():
    im.set_data(tracked_frames[0])

def animate(i):
    im.set_data(tracked_frames[i])
    return im

anim = animation.FuncAnimation(fig, animate, init_func=init, frames=len(tracked_frames),
                               interval=vid_fps*vid_config['frame_skip'])
HTML(anim.to_html5_video())

In [148]:
writer = animation.PillowWriter(fps=vid_fps/vid_config['frame_skip'])
anim.save('../assets/tracking_demo.gif', writer=writer)